In [1]:

!pip install ultralytics -q
!pip install ultralytics --upgrade -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 18.4 MB/s eta 0:00:00a 0:00:01


In [2]:
import torch, ultralytics
print(f"ultralytics: {ultralytics.__version__}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
ultralytics: 8.4.37
PyTorch: 2.10.0+cu128
CUDA: True
GPU: Tesla T4


In [3]:
import os, shutil, glob

# ---- Auto-detect dataset path ----
print("Scanning /kaggle/input/ ...")
input_path = None

for root, dirs, files in os.walk("/kaggle/input/"):
    if "data.yaml" in files and "train" in dirs and "val" in dirs:
        input_path = root
        break

if input_path is None:
    # Manual fallback — list what's in /kaggle/input/
    print("\nCould not auto-detect. Contents of /kaggle/input/:")
    for item in os.listdir("/kaggle/input/"):
        sub = os.path.join("/kaggle/input/", item)
        print(f"  {item}/")
        if os.path.isdir(sub):
            for s in os.listdir(sub)[:10]:
                print(f"    {s}")
    raise Exception("Dataset not found! Check the path above and update manually.")

print(f"Found dataset at: {input_path}")

# ---- Copy to working directory ----
WORK_DIR = "/kaggle/working/yolo_dataset"

if os.path.exists(WORK_DIR):
    shutil.rmtree(WORK_DIR)

print("Copying dataset (this may take a few minutes)...")
shutil.copytree(input_path, WORK_DIR)
print("Copy done!")

# ---- Fix data.yaml path to point to working directory ----
data_yaml = os.path.join(WORK_DIR, "data.yaml")
with open(data_yaml, 'w') as f:
    f.write(f"""# Blast Detection Dataset
path: {WORK_DIR}
train: train/images
val: val/images
test: test/images

nc: 2
names:
  0: fireball
  1: smoke_plume
""")

print(f"data.yaml updated with path: {WORK_DIR}")
print("READY!")


Scanning /kaggle/input/ ...
Found dataset at: /kaggle/input/datasets/saadhassn/blast-detection-dataset/yolo_dataset
Copying dataset (this may take a few minutes)...
Copy done!
data.yaml updated with path: /kaggle/working/yolo_dataset
READY!


In [4]:
import os
WORK_DIR = "/kaggle/working/yolo_dataset"
data_yaml = os.path.join(WORK_DIR, "data.yaml")

print("=" * 50)
for split in ['train', 'val', 'test']:
    img_dir = os.path.join(WORK_DIR, split, 'images')
    lbl_dir = os.path.join(WORK_DIR, split, 'labels')
    n_imgs = len([f for f in os.listdir(img_dir) if f.endswith(('.jpg','.png','.jpeg'))])
    n_lbls = len([f for f in os.listdir(lbl_dir) if f.endswith('.txt')])
    print(f"{split}: {n_imgs} images, {n_lbls} labels")

print("=" * 50)
with open(data_yaml) as f:
    print(f.read())


train: 15947 images, 15947 labels
val: 3417 images, 3417 labels
test: 3418 images, 3418 labels
# Blast Detection Dataset
path: /kaggle/working/yolo_dataset
train: train/images
val: val/images
test: test/images

nc: 2
names:
  0: fireball
  1: smoke_plume



In [5]:
import torch
_original_torch_load = torch.load
def _patched_torch_load(*args, **kwargs):
    kwargs['weights_only'] = False
    return _original_torch_load(*args, **kwargs)
torch.load = _patched_torch_load
print("Patched!")


Patched!


In [6]:
from ultralytics import YOLO
import os

WORK_DIR = "/kaggle/working/yolo_dataset"
data_yaml = os.path.join(WORK_DIR, "data.yaml")

# Delete old corrupted download if exists
if os.path.exists('yolov8s.pt'):
    os.remove('yolov8s.pt')

model = YOLO('yolov8s.pt')

model.train(
    data=data_yaml,
    epochs=50,
    imgsz=640,
    batch=16,
    freeze=10,
    lr0=0.001,
    lrf=0.01,
    optimizer='AdamW',
    weight_decay=0.0005,
    patience=20,
    augment=True,
    mosaic=1.0,
    mixup=0.1,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    flipud=0.1,
    fliplr=0.5,
    scale=0.5,
    device=0,
    project='/kaggle/working/runs',
    name='phase_a_frozen',
    verbose=True,
    plots=True
)

print("\nPhase A DONE!")


Ultralytics 8.4.37 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/yolo_dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.1, format=torchscript, fraction=1.0, freeze=10, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=phase_a_frozen, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, overlap_mask=True,

In [7]:
from ultralytics import YOLO
import os

WORK_DIR = "/kaggle/working/yolo_dataset"
data_yaml = os.path.join(WORK_DIR, "data.yaml")

best_a = '/kaggle/working/runs/phase_a_frozen/weights/best.pt'
model = YOLO(best_a)

model.train(
    data=data_yaml,
    epochs=100,
    imgsz=640,
    batch=16,
    freeze=0,
    lr0=0.0003,
    lrf=0.01,
    optimizer='AdamW',
    weight_decay=0.0005,
    patience=20,
    augment=True,
    mosaic=0.5,
    mixup=0.05,
    device=0,
    project='/kaggle/working/runs',
    name='phase_b_full',
    verbose=True,
    plots=True
)

print("\nPhase B DONE! Best weights: /kaggle/working/runs/phase_b_full/weights/best.pt")


Ultralytics 8.4.37 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/yolo_dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=0, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0003, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.05, mode=train, model=/kaggle/working/runs/phase_a_frozen/weights/best.pt, momentum=0.937, mosaic=0.5, multi_scale=0.0, name=phase_b_full, nbs=64, nms=False, opset=None, optimize=F

In [8]:
from ultralytics import YOLO
import os

WORK_DIR = "/kaggle/working/yolo_dataset"
data_yaml = os.path.join(WORK_DIR, "data.yaml")

best_model = '/kaggle/working/runs/phase_b_full/weights/best.pt'
model = YOLO(best_model)

metrics = model.val(data=data_yaml, split='test', imgsz=640, batch=16, verbose=True) 

print("\n" + "=" * 50)
print("FINAL TEST RESULTS")
print("=" * 50)
print(f"mAP@0.5:       {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95:  {metrics.box.map:.4f}")
print(f"Precision:      {metrics.box.mp:.4f}")
print(f"Recall:         {metrics.box.mr:.4f}")
print("=" * 50)


Ultralytics 8.4.37 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 11,126,358 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 404.6±140.6 MB/s, size: 39.8 KB)
val: Scanning /kaggle/working/yolo_dataset/test/labels... 3418 images, 1498 backgrounds, 1 corrupt: 100% ━━━━━━━━━━━━ 3418/3418 1.2Kit/s 2.8s0.0s
val: /kaggle/working/yolo_dataset/test/images/df_021318.jpg: ignoring corrupt image/label: non-normalized or out of bounds coordinates [     1.0359]
val: New cache created: /kaggle/working/yolo_dataset/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 214/214 4.5it/s 47.6s<0.2s
                   all       3417       4115      0.797      0.716      0.784      0.469
              fireball       1730       1943      0.827      0.779      0.833       0.54
           smoke_plume        859       2172      0.767      0.653      0

In [9]:
import shutil

shutil.copy2('/kaggle/working/runs/phase_b_full/weights/best.pt', '/kaggle/working/best.pt')
shutil.copy2('/kaggle/working/runs/phase_a_frozen/weights/best.pt', '/kaggle/working/best_phase_a.pt')

print("DOWNLOAD THESE FROM THE 'Output' TAB:")
print("  1. best.pt          <-- YOUR FINAL MODEL (use this)")
print("  2. best_phase_a.pt  <-- Backup")


DOWNLOAD THESE FROM THE 'Output' TAB:
  1. best.pt          <-- YOUR FINAL MODEL (use this)
  2. best_phase_a.pt  <-- Backup
